# 07 A/B Test Evaluation

This notebook is the seventh step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

The goal of this stage is to simulate an A/B test and evaluate whether the intent-aware recommendation strategy improves lead generation and commercial outcomes compared with the popularity-based baseline.

In the previous notebook, I designed three recommendation strategies:

1. Popularity-based recommendation;
2. Category preference recommendation;
3. Intent-aware recommendation.

In this notebook, I focus on the main experiment:

- **Control group:** popularity-based recommendation;
- **Treatment group:** intent-aware recommendation.

The evaluation metrics include:

- Click-through rate;
- Inquiry rate;
- Purchase rate;
- Conversion rate;
- GMV;
- Revenue per user;
- Uplift;
- Statistical significance.

This notebook simulates post-launch evaluation for a recommendation strategy product experiment.

## Step 1: Import Libraries and Define Project Paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Define project paths
BASE_DIR = Path("..")

FINAL_DIR = BASE_DIR / "data" / "final"
OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"

# Create folders if they do not exist
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project paths defined successfully.")

Libraries imported and project paths defined successfully.


## Step 2: Load Recommendation Dataset
For the main A/B test, I compare:

- popularity-based recommendation as the control group;

- intent-aware recommendation as the treatment group.

In [2]:
# Load recommendation dataset
fact_recommendations = pd.read_csv(FINAL_DIR / "fact_recommendations.csv")

# Convert date column if available
if "recommendation_date" in fact_recommendations.columns:
    fact_recommendations["recommendation_date"] = pd.to_datetime(
        fact_recommendations["recommendation_date"]
    )

print("Recommendation dataset loaded successfully.")
print("Dataset shape:", fact_recommendations.shape)

fact_recommendations.head()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Recommendation dataset loaded successfully.
Dataset shape: (15000, 17)


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate,experiment_group,recommendation_date
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,control,2026-01-01
1,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,category_preference,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_category,2026-01-01
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.877,3.8415,0.097,99.55,1.0,treatment_intent,2026-01-01
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.877,3.8415,0.097,99.93,1.0,control,2026-01-01
4,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,category_preference,3986c6e32f39114eff1ded07f1cb16fb,744dac408745240a2c2528fb1b6028f3,industry_commerce_and_business,0.2079,748.00,5.0000,1.000,4.5542,0.012,99.93,1.0,treatment_category,2026-01-01


## Step 3: Prepare Control and Treatment Groups

For this experiment, I compare two recommendation strategies:

### Control Group

- `popularity_based`
- Represents the baseline recommendation logic.

### Treatment Group

- `intent_aware`
- Represents the AI-assisted recommendation strategy that uses lead score, product quality, seller quality, category match, and user segment.

The category preference strategy is excluded from the main A/B comparison but can be used as an additional benchmark later.

In [3]:
# Keep only control and treatment groups for main A/B test
experiment_base = fact_recommendations[
    fact_recommendations["strategy"].isin(["popularity_based", "intent_aware"])
].copy()

# Map experiment group labels
experiment_base["experiment_group"] = np.where(
    experiment_base["strategy"] == "popularity_based",
    "control",
    "treatment"
)

print("Experiment base shape:", experiment_base.shape)
print(experiment_base["experiment_group"].value_counts())

experiment_base.head()

Experiment base shape: (10000, 17)
experiment_group
control      5000
treatment    5000
Name: count, dtype: int64


,user_id,user_value_segment,preferred_category,strategy,recommended_product_id,recommended_seller_id,recommended_category,recommendation_score,product_avg_price,product_avg_review_score,product_high_intent_rate,seller_avg_review_score,seller_late_delivery_rate,user_avg_model_lead_score,user_high_intent_rate,experiment_group,recommendation_date
0,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.55,1.0,control,2026-01-01
2,842bff27a9fd73c774aeb526d0f53113,Low Value,bed_bath_table,intent_aware,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,1.0309,88.15,3.9444,0.8770,3.8415,0.0970,99.55,1.0,treatment,2026-01-01
3,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.93,1.0,control,2026-01-01
5,97e5be9a2f1841bc42b966aa2d014d13,Medium Value,industry_commerce_and_business,intent_aware,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,furniture_decor,0.9624,71.35,4.0538,0.9108,4.0944,0.0645,99.93,1.0,treatment,2026-01-01
6,987f4716cef1294c01c4c0cb7b275ff1,High Value,bed_bath_table,popularity_based,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,bed_bath_table,0.8569,88.15,3.9444,0.8770,3.8415,0.0970,99.83,1.0,control,2026-01-01


## Step 4: Define Experiment Outcome Simulation Logic

Because this is a portfolio project using public historical data, I do not have real online A/B test outcomes.

Therefore, I simulate experiment outcomes based on recommendation quality signals.

The simulated outcomes include:

- clicked;
- inquired;
- purchased;
- revenue.

The simulation uses business-driven assumptions:

1. Higher recommendation score increases click probability;
2. Higher product review score increases user trust;
3. Higher product high-intent rate increases inquiry and purchase probability;
4. Higher seller review score improves conversion;
5. Higher seller late delivery rate reduces conversion;
6. Higher user lead score increases engagement and purchase probability;
7. Intent-aware recommendations receive a moderate treatment lift to represent better matching.

This simulation is designed for demonstrating A/B test evaluation workflow, not for claiming real-world causal impact.

In [6]:
def clip_probability(p, lower=0.01, upper=0.95):
    """
    Clip probability values to avoid unrealistic 0 or 1 probabilities.
    """
    return np.minimum(np.maximum(p, lower), upper)


def simulate_experiment_outcomes(row):
    """
    Simulate clicked, inquired, purchased, and revenue outcomes for each recommendation.
    
    The simulation is based on recommendation quality, product quality, seller quality,
    user lead score, and treatment effect.
    """
    # Base probabilities
    base_click_prob = 0.08
    base_inquiry_prob = 0.035
    base_purchase_prob = 0.018
    
    # Normalised signals
    recommendation_score = row["recommendation_score"]
    product_review = row["product_avg_review_score"] / 5
    product_intent = row["product_high_intent_rate"]
    seller_review = row["seller_avg_review_score"] / 5
    seller_late_penalty = row["seller_late_delivery_rate"]
    user_lead_score = row["user_avg_model_lead_score"] / 100
    
    # Treatment lift for intent-aware recommendation
    treatment_lift = 1.18 if row["experiment_group"] == "treatment" else 1.00
    
    # User segment adjustment
    if row["user_value_segment"] == "High Value":
        segment_lift = 1.15
    elif row["user_value_segment"] == "Medium Value":
        segment_lift = 1.07
    else:
        segment_lift = 1.00
    
    # Click probability
    click_prob = (
        base_click_prob
        + 0.06 * recommendation_score
        + 0.03 * product_review
        + 0.02 * user_lead_score
    ) * treatment_lift * segment_lift
    
    # Inquiry probability
    inquiry_prob = (
        base_inquiry_prob
        + 0.04 * product_intent
        + 0.03 * user_lead_score
        + 0.02 * seller_review
        - 0.02 * seller_late_penalty
    ) * treatment_lift * segment_lift
    
    # Purchase probability
    purchase_prob = (
        base_purchase_prob
        + 0.035 * product_intent
        + 0.025 * user_lead_score
        + 0.015 * seller_review
        - 0.018 * seller_late_penalty
    ) * treatment_lift * segment_lift
    
    # Clip probabilities
    click_prob = clip_probability(click_prob, 0.01, 0.60)
    inquiry_prob = clip_probability(inquiry_prob, 0.005, 0.45)
    purchase_prob = clip_probability(purchase_prob, 0.003, 0.35)
    
    # Simulate outcomes
    clicked = np.random.binomial(1, click_prob)
    
    # Inquiry is more likely if clicked
    if clicked == 1:
        inquired = np.random.binomial(1, clip_probability(inquiry_prob * 1.6, 0.005, 0.60))
    else:
        inquired = np.random.binomial(1, inquiry_prob)
    
    # Purchase is more likely if clicked or inquired
    purchase_multiplier = 1.0
    if clicked == 1:
        purchase_multiplier += 0.6
    if inquired == 1:
        purchase_multiplier += 0.9
    
    purchased = np.random.binomial(
        1,
        clip_probability(purchase_prob * purchase_multiplier, 0.003, 0.70)
    )
    
    # Revenue only occurs if purchased
    revenue = row["product_avg_price"] if purchased == 1 else 0
    
    return pd.Series({
        "click_probability": round(click_prob, 4),
        "inquiry_probability": round(inquiry_prob, 4),
        "purchase_probability": round(purchase_prob, 4),
        "clicked": clicked,
        "inquired": inquired,
        "purchased": purchased,
        "revenue": round(revenue, 2)
    })

## Step 5: Simulate Experiment Outcomes

I apply the simulation logic to each recommendation record.

The output becomes a simulated experiment table where each row represents one user recommendation exposure and its outcome.

In [21]:
# Simulate outcomes
simulated_outcomes = experiment_base.apply(
    simulate_experiment_outcomes,
    axis=1
).reset_index(drop=True)

# Reset experiment_base index as well
experiment_base_reset = experiment_base.reset_index(drop=True)

# Combine with experiment base
fact_recommendation_experiment = pd.concat(
    [experiment_base_reset, simulated_outcomes],
    axis=1
)

print("Experiment dataset shape:", fact_recommendation_experiment.shape)

print("\nMissing values in outcome columns:")
print(fact_recommendation_experiment[[
    "click_probability",
    "inquiry_probability",
    "purchase_probability",
    "clicked",
    "inquired",
    "purchased",
    "revenue"
]].isna().sum())

fact_recommendation_experiment[[
    "user_id",
    "experiment_group",
    "strategy",
    "recommended_product_id",
    "recommendation_score",
    "clicked",
    "inquired",
    "purchased",
    "revenue"
]].head()

Experiment dataset shape: (10000, 24)

Missing values in outcome columns:
click_probability       0
inquiry_probability     0
purchase_probability    0
clicked                 0
inquired                0
purchased               0
revenue                 0
dtype: int64


,user_id,experiment_group,strategy,recommended_product_id,recommendation_score,clicked,inquired,purchased,revenue
0,842bff27a9fd73c774aeb526d0f53113,control,popularity_based,99a4788cb24856965c36a24e339b6058,0.8569,0.0,0.0,0.0,0.0
1,842bff27a9fd73c774aeb526d0f53113,treatment,intent_aware,99a4788cb24856965c36a24e339b6058,1.0309,0.0,0.0,0.0,0.0
2,97e5be9a2f1841bc42b966aa2d014d13,control,popularity_based,99a4788cb24856965c36a24e339b6058,0.8569,0.0,0.0,0.0,0.0
3,97e5be9a2f1841bc42b966aa2d014d13,treatment,intent_aware,aca2eb7d00ea1a7b8ebd4e68314663af,0.9624,1.0,1.0,0.0,0.0
4,987f4716cef1294c01c4c0cb7b275ff1,control,popularity_based,99a4788cb24856965c36a24e339b6058,0.8569,0.0,0.0,0.0,0.0


## Step 6: Validate Simulated Outcome Distribution

Before evaluating the A/B test, I inspect the distribution of simulated outcomes.

This helps ensure that the simulated experiment results are reasonable and not overly extreme.

In [22]:
outcome_distribution = (
    fact_recommendation_experiment
    .groupby("experiment_group")
    .agg(
        exposures=("user_id", "count"),
        clicked=("clicked", "sum"),
        inquired=("inquired", "sum"),
        purchased=("purchased", "sum"),
        total_revenue=("revenue", "sum"),
        avg_revenue=("revenue", "mean")
    )
    .reset_index()
)

outcome_distribution["ctr"] = (
    outcome_distribution["clicked"] / outcome_distribution["exposures"]
).round(4)

outcome_distribution["inquiry_rate"] = (
    outcome_distribution["inquired"] / outcome_distribution["exposures"]
).round(4)

outcome_distribution["purchase_rate"] = (
    outcome_distribution["purchased"] / outcome_distribution["exposures"]
).round(4)

outcome_distribution["revenue_per_user"] = (
    outcome_distribution["total_revenue"] / outcome_distribution["exposures"]
).round(4)

outcome_distribution["total_revenue"] = outcome_distribution["total_revenue"].round(2)
outcome_distribution["avg_revenue"] = outcome_distribution["avg_revenue"].round(4)

outcome_distribution

,experiment_group,exposures,clicked,inquired,purchased,total_revenue,avg_revenue,ctr,inquiry_rate,purchase_rate,revenue_per_user
0,control,5000,971.0,635.0,516.0,45485.40,9.0971,0.1942,0.1270,0.1032,9.0971
1,treatment,5000,1138.0,816.0,678.0,65898.25,13.1796,0.2276,0.1632,0.1356,13.1796


## Step 7: Create A/B Test Summary

This summary calculates the core A/B testing metrics by experiment group.

Metrics:

- Exposures;
- Clicks;
- Inquiries;
- Purchases;
- CTR;
- Inquiry rate;
- Purchase rate;
- Total revenue;
- Revenue per user.

This table provides the main post-launch evaluation view.

In [23]:
ab_test_summary = outcome_distribution.copy()

ab_test_summary = ab_test_summary[[
    "experiment_group",
    "exposures",
    "clicked",
    "inquired",
    "purchased",
    "ctr",
    "inquiry_rate",
    "purchase_rate",
    "total_revenue",
    "revenue_per_user"
]]

ab_test_summary

,experiment_group,exposures,clicked,inquired,purchased,ctr,inquiry_rate,purchase_rate,total_revenue,revenue_per_user
0,control,5000,971.0,635.0,516.0,0.1942,0.1270,0.1032,45485.40,9.0971
1,treatment,5000,1138.0,816.0,678.0,0.2276,0.1632,0.1356,65898.25,13.1796


In [24]:
# Export A/B test summary
ab_test_summary.to_csv(
    TABLE_OUTPUT_DIR / "ab_test_summary.csv",
    index=False
)

print("ab_test_summary.csv exported successfully.")

ab_test_summary.csv exported successfully.


## Step 8: Calculate Treatment Uplift

Uplift measures the percentage improvement of the treatment group compared with the control group.

Formula:

\[
Uplift = \frac{Treatment - Control}{Control}
\]

I calculate uplift for:

- CTR;
- Inquiry rate;
- Purchase rate;
- Revenue per user;
- Total revenue.

In [25]:
# Extract control and treatment metrics
control_metrics = ab_test_summary[
    ab_test_summary["experiment_group"] == "control"
].iloc[0]

treatment_metrics = ab_test_summary[
    ab_test_summary["experiment_group"] == "treatment"
].iloc[0]

def calculate_uplift(control_value, treatment_value):
    """
    Calculate percentage uplift from control to treatment.
    """
    if control_value == 0:
        return np.nan
    return (treatment_value - control_value) / control_value

uplift_records = []

for metric in [
    "ctr",
    "inquiry_rate",
    "purchase_rate",
    "revenue_per_user",
    "total_revenue"
]:
    uplift_records.append({
        "metric": metric,
        "control_value": control_metrics[metric],
        "treatment_value": treatment_metrics[metric],
        "absolute_difference": round(treatment_metrics[metric] - control_metrics[metric], 4),
        "uplift_rate": round(calculate_uplift(control_metrics[metric], treatment_metrics[metric]), 4)
    })

ab_test_uplift_summary = pd.DataFrame(uplift_records)

ab_test_uplift_summary

,metric,control_value,treatment_value,absolute_difference,uplift_rate
0,ctr,0.1942,0.2276,0.0334,0.1720
1,inquiry_rate,0.1270,0.1632,0.0362,0.2850
2,purchase_rate,0.1032,0.1356,0.0324,0.3140
3,revenue_per_user,9.0971,13.1796,4.0825,0.4488
4,total_revenue,45485.4000,65898.2500,20412.8500,0.4488


In [26]:
# Export uplift summary
ab_test_uplift_summary.to_csv(
    TABLE_OUTPUT_DIR / "ab_test_uplift_summary.csv",
    index=False
)

print("ab_test_uplift_summary.csv exported successfully.")

ab_test_uplift_summary.csv exported successfully.


## Step 9: Statistical Significance Testing

I conduct statistical tests to check whether treatment-control differences are statistically meaningful.

For binary metrics such as clicked, inquired, and purchased, I use a two-proportion z-test.

For revenue per user, I use an independent two-sample t-test.

Metrics tested:

- CTR;
- Inquiry rate;
- Purchase rate;
- Revenue per user.

Important note: Because this is a simulated portfolio project, statistical significance should be interpreted as a demonstration of A/B test evaluation workflow rather than evidence of real online causal impact.

In [27]:
def two_proportion_z_test(success_control, n_control, success_treatment, n_treatment):
    """
    Perform a two-proportion z-test.
    """
    p_control = success_control / n_control
    p_treatment = success_treatment / n_treatment
    
    pooled_p = (success_control + success_treatment) / (n_control + n_treatment)
    
    standard_error = np.sqrt(
        pooled_p * (1 - pooled_p) * (1 / n_control + 1 / n_treatment)
    )
    
    if standard_error == 0:
        return np.nan, np.nan
    
    z_score = (p_treatment - p_control) / standard_error
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    return round(z_score, 4), round(p_value, 6)


# Split control and treatment data
control_data = fact_recommendation_experiment[
    fact_recommendation_experiment["experiment_group"] == "control"
]

treatment_data = fact_recommendation_experiment[
    fact_recommendation_experiment["experiment_group"] == "treatment"
]

n_control = len(control_data)
n_treatment = len(treatment_data)

significance_records = []

# Binary outcome tests
for metric, success_col in [
    ("ctr", "clicked"),
    ("inquiry_rate", "inquired"),
    ("purchase_rate", "purchased")
]:
    z_score, p_value = two_proportion_z_test(
        control_data[success_col].sum(),
        n_control,
        treatment_data[success_col].sum(),
        n_treatment
    )
    
    significance_records.append({
        "metric": metric,
        "test_type": "two_proportion_z_test",
        "test_statistic": z_score,
        "p_value": p_value,
        "significant_at_0_05": p_value < 0.05 if not pd.isna(p_value) else False
    })

# Revenue per user t-test
t_stat, p_value = stats.ttest_ind(
    treatment_data["revenue"],
    control_data["revenue"],
    equal_var=False
)

significance_records.append({
    "metric": "revenue_per_user",
    "test_type": "welch_t_test",
    "test_statistic": round(t_stat, 4),
    "p_value": round(p_value, 6),
    "significant_at_0_05": p_value < 0.05
})

ab_test_significance = pd.DataFrame(significance_records)

ab_test_significance

,metric,test_type,test_statistic,p_value,significant_at_0_05
0,ctr,two_proportion_z_test,4.0937,0.000042,True
1,inquiry_rate,two_proportion_z_test,5.1391,0.000000,True
2,purchase_rate,two_proportion_z_test,4.9960,0.000001,True
3,revenue_per_user,welch_t_test,5.8613,0.000000,True


In [28]:
# Export significance test results
ab_test_significance.to_csv(
    TABLE_OUTPUT_DIR / "ab_test_significance.csv",
    index=False
)

print("ab_test_significance.csv exported successfully.")

ab_test_significance.csv exported successfully.


## Step 10: Segment-Level A/B Test Analysis

Segment-level analysis helps determine whether the treatment effect differs across user groups.

In this step, I evaluate A/B test performance by:

- User value segment;
- Recommended category.

This provides more actionable product insight than only looking at the overall average effect.

In [29]:
# Segment-level summary by user value segment
ab_test_segment_summary = (
    fact_recommendation_experiment
    .groupby(["user_value_segment", "experiment_group"])
    .agg(
        exposures=("user_id", "count"),
        clicked=("clicked", "sum"),
        inquired=("inquired", "sum"),
        purchased=("purchased", "sum"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
)

ab_test_segment_summary["ctr"] = (
    ab_test_segment_summary["clicked"] / ab_test_segment_summary["exposures"]
).round(4)

ab_test_segment_summary["inquiry_rate"] = (
    ab_test_segment_summary["inquired"] / ab_test_segment_summary["exposures"]
).round(4)

ab_test_segment_summary["purchase_rate"] = (
    ab_test_segment_summary["purchased"] / ab_test_segment_summary["exposures"]
).round(4)

ab_test_segment_summary["revenue_per_user"] = (
    ab_test_segment_summary["total_revenue"] / ab_test_segment_summary["exposures"]
).round(4)

ab_test_segment_summary["total_revenue"] = ab_test_segment_summary["total_revenue"].round(2)

ab_test_segment_summary

,user_value_segment,experiment_group,exposures,clicked,inquired,purchased,total_revenue,ctr,inquiry_rate,purchase_rate,revenue_per_user
0,High Value,control,1625,331.0,237.0,183.0,16131.45,0.2037,0.1458,0.1126,9.9270
1,High Value,treatment,1625,406.0,285.0,231.0,22688.30,0.2498,0.1754,0.1422,13.9620
2,Low Value,control,1621,293.0,184.0,154.0,13575.10,0.1808,0.1135,0.0950,8.3745
3,Low Value,treatment,1621,334.0,233.0,194.0,18373.88,0.2060,0.1437,0.1197,11.3349
4,Medium Value,control,1652,330.0,205.0,173.0,15249.95,0.1998,0.1241,0.1047,9.2312
5,Medium Value,treatment,1652,377.0,285.0,242.0,23393.67,0.2282,0.1725,0.1465,14.1608
6,Unknown,control,102,17.0,9.0,6.0,528.90,0.1667,0.0882,0.0588,5.1853
7,Unknown,treatment,102,21.0,13.0,11.0,1442.40,0.2059,0.1275,0.1078,14.1412


In [30]:
# Export segment-level A/B test summary
ab_test_segment_summary.to_csv(
    TABLE_OUTPUT_DIR / "ab_test_segment_summary.csv",
    index=False
)

print("ab_test_segment_summary.csv exported successfully.")

ab_test_segment_summary.csv exported successfully.


## Step 11: Category-Level A/B Test Analysis

Category-level analysis helps identify which product categories benefit most from the intent-aware recommendation strategy.

This is useful for recommendation strategy iteration because treatment effects may vary by product category.

In [31]:
# Category-level summary
ab_test_category_summary = (
    fact_recommendation_experiment
    .groupby(["recommended_category", "experiment_group"])
    .agg(
        exposures=("user_id", "count"),
        clicked=("clicked", "sum"),
        inquired=("inquired", "sum"),
        purchased=("purchased", "sum"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
)

ab_test_category_summary["ctr"] = (
    ab_test_category_summary["clicked"] / ab_test_category_summary["exposures"]
).round(4)

ab_test_category_summary["inquiry_rate"] = (
    ab_test_category_summary["inquired"] / ab_test_category_summary["exposures"]
).round(4)

ab_test_category_summary["purchase_rate"] = (
    ab_test_category_summary["purchased"] / ab_test_category_summary["exposures"]
).round(4)

ab_test_category_summary["revenue_per_user"] = (
    ab_test_category_summary["total_revenue"] / ab_test_category_summary["exposures"]
).round(4)

ab_test_category_summary["total_revenue"] = ab_test_category_summary["total_revenue"].round(2)

ab_test_category_summary = ab_test_category_summary.sort_values(
    by=["recommended_category", "experiment_group"]
)

ab_test_category_summary.head(20)

,recommended_category,experiment_group,exposures,clicked,inquired,purchased,total_revenue,ctr,inquiry_rate,purchase_rate,revenue_per_user
0,bed_bath_table,control,4981,965.0,633.0,516.0,45485.40,0.1937,0.1271,0.1036,9.1318
1,bed_bath_table,treatment,456,112.0,73.0,60.0,5289.00,0.2456,0.1601,0.1316,11.5987
2,computers_accessories,treatment,342,89.0,47.0,42.0,5808.81,0.2602,0.1374,0.1228,16.9848
3,furniture_decor,control,19,6.0,2.0,0.0,0.00,0.3158,0.1053,0.0000,0.0000
4,furniture_decor,treatment,3347,733.0,563.0,463.0,33035.05,0.2190,0.1682,0.1383,9.8700
5,garden_tools,treatment,150,39.0,26.0,25.0,1371.95,0.2600,0.1733,0.1667,9.1463
6,health_beauty,treatment,423,88.0,60.0,48.0,15726.24,0.2080,0.1418,0.1135,37.1779
7,watches_gifts,treatment,282,77.0,47.0,40.0,4667.20,0.2730,0.1667,0.1418,16.5504


In [32]:
# Export category-level A/B test summary
ab_test_category_summary.to_csv(
    TABLE_OUTPUT_DIR / "ab_test_category_summary.csv",
    index=False
)

print("ab_test_category_summary.csv exported successfully.")

ab_test_category_summary.csv exported successfully.


## Step 12: Create Final Experiment Dataset

The final experiment dataset contains one row per recommendation exposure and includes:

- Experiment group;
- Recommendation strategy;
- Recommended product and category;
- User segment;
- Recommendation quality signals;
- Simulated outcome probabilities;
- Simulated user outcomes.

This table will support dashboard development and final reporting.

In [33]:
# Export final experiment dataset
fact_recommendation_experiment.to_csv(
    FINAL_DIR / "fact_recommendation_experiment.csv",
    index=False
)

print("fact_recommendation_experiment.csv exported successfully.")
print("File saved to:", FINAL_DIR / "fact_recommendation_experiment.csv")

fact_recommendation_experiment.csv exported successfully.
File saved to: ../data/final/fact_recommendation_experiment.csv


## Step 13: Business Interpretation

The A/B test simulation provides a structured way to evaluate recommendation strategy performance.

The key interpretation logic is:

1. If treatment has higher CTR, the intent-aware recommendation is more effective at attracting user attention.
2. If treatment has higher inquiry rate, it is better at generating lead intent.
3. If treatment has higher purchase rate or CVR, it is better at converting users.
4. If treatment has higher revenue per user, it creates stronger commercial value.
5. If uplift is statistically significant, the observed difference is less likely to be random under the simulation assumptions.

However, because this is a simulated portfolio project, the results should be interpreted as a demonstration of experiment design and post-launch evaluation workflow rather than real production causal evidence.

## Step 14: Summary of A/B Test Evaluation

In this notebook, I simulated an A/B test to compare a popularity-based recommendation baseline with an intent-aware recommendation strategy.

Key outputs generated:

- `fact_recommendation_experiment.csv`
- `ab_test_summary.csv`
- `ab_test_uplift_summary.csv`
- `ab_test_significance.csv`
- `ab_test_segment_summary.csv`
- `ab_test_category_summary.csv`

Business value of this stage:

1. Designed a control-treatment experiment structure;
2. Simulated user outcomes including click, inquiry, purchase, and revenue;
3. Evaluated recommendation performance using CTR, inquiry rate, purchase rate, GMV, and revenue per user;
4. Calculated treatment uplift against the popularity-based baseline;
5. Conducted statistical significance testing;
6. Produced segment-level and category-level insights for recommendation strategy iteration.

This stage demonstrates how AI-assisted recommendation strategies can be evaluated through product experimentation and post-launch analysis.